### 0.Import & Data Load

In [35]:
%pip install lightgbm

Note: you may need to restart the kernel to use updated packages.


In [36]:
from pathlib import Path

import numpy as np
import pandas as pd

from lightgbm import LGBMRegressor
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score

In [37]:
DATA_DIR = Path('/Users/bitsaem/Desktop/stress_project/open')

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

#### Check Data

In [38]:
train.isnull().sum()

ID                             0
gender                         0
age                            0
height                         0
weight                         0
cholesterol                    0
systolic_blood_pressure        0
diastolic_blood_pressure       0
glucose                        0
bone_density                   0
activity                       0
smoke_status                   0
medical_history             1289
family_medical_history      1486
sleep_pattern                  0
edu_level                    607
mean_working                1032
stress_score                   0
dtype: int64

In [39]:
# 결측값 있는 칼럼(column) 확인
missing_columns_train = train.columns[train.isnull().sum() > 0]
missing_columns_train

Index(['medical_history', 'family_medical_history', 'edu_level',
       'mean_working'],
      dtype='object')

In [40]:
train[missing_columns_train].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   medical_history         1711 non-null   object 
 1   family_medical_history  1514 non-null   object 
 2   edu_level               2393 non-null   object 
 3   mean_working            1968 non-null   float64
dtypes: float64(1), object(3)
memory usage: 93.9+ KB


In [41]:
categorical_na_cols = []
numerical_na_cols = []

# 결측값이 있는 각 칼럼에 대해 데이터 타입 확인
for col in missing_columns_train:
    # 해당 칼럼이 범주형(object 또는 category)인지 확인
    if col in train.select_dtypes(include=['object', 'category']).columns:
        categorical_na_cols.append(col)
    # 해당 칼럼이 수치형(int 또는 float)인지 확인
    elif col in train.select_dtypes(include=['int', 'float']).columns:
        numerical_na_cols.append(col)

print("결측값이 있는 범주형 변수:", categorical_na_cols)
print("결측값이 있는 수치형 변수:", numerical_na_cols)

결측값이 있는 범주형 변수: ['medical_history', 'family_medical_history', 'edu_level']
결측값이 있는 수치형 변수: ['mean_working']


### 1. Data Preprocessing

In [42]:
TARGET = 'stress_score'
ID_COL = 'ID'

X = train.drop(columns=[TARGET, ID_COL])
y = train[TARGET].copy()

X_test = test.drop(columns=[ID_COL])
test_ids = test[ID_COL].copy()

categorical_cols = X.select_dtypes(
    include=['object', 'category']
).columns.tolist()

numerical_cols = X.select_dtypes(
    include=['number']
).columns.tolist()

print('범주형 변수:', categorical_cols)
print('수치형 변수:', numerical_cols)

범주형 변수: ['gender', 'activity', 'smoke_status', 'medical_history', 'family_medical_history', 'sleep_pattern', 'edu_level']
수치형 변수: ['age', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure', 'diastolic_blood_pressure', 'glucose', 'bone_density', 'mean_working']


In [43]:
categorical_pipeline = Pipeline([
    (
        'imputer',
        SimpleImputer(
            strategy='constant',
            fill_value='missing'
        )
    ),
    (
        'encoder',
        OneHotEncoder(
            handle_unknown='ignore',
            sparse_output=True
        )
    )
])

numerical_pipeline = Pipeline([
    (
        'imputer',
        SimpleImputer(
            strategy='median',
            add_indicator=True
        )
    )
])

preprocessor = ColumnTransformer([
    ('categorical', categorical_pipeline, categorical_cols),
    ('numerical', numerical_pipeline, numerical_cols)
])

In [44]:
model = LGBMRegressor(
    objective='regression_l1',
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    random_state=42,
    verbosity=-1
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
])

In [45]:
X = train.drop(columns=['stress_score', 'ID'])
y = train['stress_score'].copy()

X_test = test.drop(columns=['ID'])
test_ids = test['ID'].copy()

### 2. Train / Predict

In [46]:
model = LGBMRegressor(
    objective='regression_l1',
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    random_state=42,
    verbosity=-1
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
])

pipeline.fit(X, y)

pred = pipeline.predict(X_test)
pred = np.clip(pred, 0, 1)

In [47]:
# 모델정의
model = LGBMRegressor(
    objective='regression_l1',
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    random_state=42,
    verbosity=-1
)
# 전처리+모델 연결
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
])

In [48]:
# 교차검증
cv = KFold(n_splits=5, shuffle=True, random_state=42)

negative_mae = cross_val_score(
    pipeline,
    X,
    y,
    cv=cv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)

fold_mae = -negative_mae

print('Fold별 MAE:', np.round(fold_mae, 6))
print('평균 CV MAE:', round(fold_mae.mean(), 6))
print('MAE 표준편차:', round(fold_mae.std(), 6))

Fold별 MAE: [0.205547 0.205694 0.214054 0.223545 0.210072]
평균 CV MAE: 0.211783
MAE 표준편차: 0.006669


In [49]:
# train 데이터 최종학습
pipeline.fit(X, y)

# test 예측
pred = pipeline.predict(X_test)
pred = np.clip(pred, 0, 1)

### Submission

In [50]:
submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

In [51]:
submission['stress_score'] = pred
submission.head()

,ID,stress_score
0,TEST_0000,0.425558
1,TEST_0001,0.609713
2,TEST_0002,0.228139
3,TEST_0003,0.487783
4,TEST_0004,0.550890


In [52]:
submission.to_csv('submit_v1.csv', index=False)